### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following
1. Tracking agent behaviour with logging, analytics and debugging
2. Transforming prompts, tool selection and output formatting
3. Adding retries, fallbacks and early termination logic
4. Applying rate limits, guardrails and Pii detection

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Real World Example. - Airport Security 
1. passenger -> Security Check (Middleware 1) -> Immigration (Middleware 2) -> Boarding (Middleware 3) -> Enter Flight 

Built In Middlewares
1. Summariazation -> Agent -> Summarizes the List of the messages on a certain count
2. Human in the feedback
3. Model call limit -> Limit the Modal calls to prevent excessive cost
4. PII Detection -> Detect and handle Personally Identifiable Information

### Summarization Middleware

Automatically summarize the convesation history when approching token limits, preserving recent messages while compressing older context. Summarization is useful for the following : 
1. Long-running conversations that exceed context window
2. Multi-turn dialouges with extensive history
3. Applications where preserving full conservation context matters

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

# Messagebased Summarization
agent = create_agent(
    model="gpt-4o",
    checkpointer = InMemorySaver(),
    middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger=("messages",10),    
                keep=("messages", 4)
            )
        ]
)    


In [5]:
# Run with a thread ID
config = {"configurable" : {"thread_id": "test-1"} }

In [11]:
# Altenative test data
questions = [
    "What is 2+2?",
    "What is 3+3?",
    "What is 4+4?",
    "What is 5+5?",
    "What is 6+6?",
    "What is 7+7?",
    "What is 8+8?",
    "What is 9+9?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    #print(f"Messages : {response['messages'][-1].content}")
    print(f"Messages : {response}")
    print(f"Messages : {len(response['messages'])}")

Messages : {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is asking simple arithmetic questions to confirm basic addition operations.\n\n## SUMMARY\nThe user has asked a series of addition questions and received responses:\n1. "What is 2+2?" - Answered with 4.\n2. "What is 3+3?" - Answered with 6.\n3. "What is 4+4?" - Answered with 8.\n4. "What is 5+5?" - Answered with 10.\n5. "What is 6+6?" - Answered with 12.\n6. "What is 7+7?" - Answered with 14.\n7. "What is 8+8?" - Answered with 16.\n8. "What is 9+9?" - Answered with 18.\n9. The user has repeated the questions "What is 2+2?" and "What is 3+3?" again.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nRespond to the user\'s repeated questions, "What is 2+2?" and "What is 3+3?" next.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e59072cf-abc6-4410-9d07-4988bc7de3ca'), AIMessage(content='6 + 6 equals 12.', additional_kwargs={'refusal': None}, resp

### Token Size

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool


@tool
def search_hotels(city:str) -> str:
    """Search for hotels in a given city."""
    return f"Found 5 hotels in {city} : Hotel A, Hotel B, Hotel C, Hotel D, Hotel E"

# Messagebased Summarization
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger=("tokens",300),    
                keep=("tokens", 200)
            )
        ]
)    

config = {"configurable" : {"thread_id": "test-2"} }

def count_tokens(messages):
    total_chars = sum(len(message.content) for message in messages)
    return total_chars // 4  # Rough estimate: 1 token ≈ 4 characters

In [18]:
# Run test
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Search for hotels in {city}")]},config=config)
    print(f"Response : {response}")
    print(f"Messages : {len(response['messages'])}")
    print(f"Tokens : {count_tokens(response['messages'])}")

Response : {'messages': [HumanMessage(content='Search for hotels in New York', additional_kwargs={}, response_metadata={}, id='1fb23af2-7fd1-416c-abd7-5ec807bae062'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 52, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6c73b7f049', 'id': 'chatcmpl-EbNz7NBz2raNU4gNgfqU28ozDtiWN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15e09-7801-71ce-a359-385f36975a56-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_E9flSxLP0u

### Fraction 

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool


@tool
def search_hotels(city:str) -> str:
    """Search for hotels in a given city."""
    return f"Found 5 hotels in {city} : Hotel A, Hotel B, Hotel C, Hotel D, Hotel E"

# Messagebased Summarization
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger=("fraction",0.005),   # 0.5% = 640 tokens
                keep=("tokens", 0.002) # 0.2% = 256 tokens
            )
        ]
)    

config = {"configurable" : {"thread_id": "test-2"} }

def count_tokens(messages):
    total_chars = sum(len(message.content) for message in messages)
    return total_chars // 4  # Rough estimate: 1 token ≈ 4 characters

In [20]:
# Run test
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Search for hotels in {city}")]},config=config)
    print(f"Response : {response}")
    print(f"Messages : {len(response['messages'])}")
    print(f"Tokens : {count_tokens(response['messages'])}")

Response : {'messages': [HumanMessage(content='Search for hotels in New York', additional_kwargs={}, response_metadata={}, id='bab3c688-0419-4c71-a381-e48894571c18'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 52, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6c73b7f049', 'id': 'chatcmpl-EbO1mkqMwqATXs4keptFv2AgPsp9f', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15e0b-f9f0-73e8-8b3d-23999fcb97f7-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_DyPGkvTc6x

### Human in the Loop Middleware

Pause Agent Execution for human approval, editing or rejection of tool calls before they execute. Human in the loop is useful for the following
1. High-stakes operations requiring human approval (e.g., database writes, financial transactions)
2. Compliance workflows where human oversight is mandatory
3. Long-runnning conversations where human feedback guides the agent

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

In [2]:
def read_email_tool(email_id:str) -> str:
    """Read an email by its ID."""
    return f"Email content for ID {email_id}: This is a sample email content."

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Send an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'."

In [ ]:
agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool": {
                        "allowed_decisions": ["approve", "reject","edit"]
                    },
                    "read_email_tool": False,
                }
            )
        ]
)

In [24]:
config = {"configurable" : {"thread_id": "test-4"} }
result = agent.invoke(
    {"messages":[HumanMessage(
    content="send email to john@example.com with subject 'Hello' and body 'This is a test email.'")]},
    config=config
    )
print(f"Response : {result}")

Response : {'messages': [HumanMessage(content="send email to john@example.com with subject 'Hello' and body 'This is a test email.'", additional_kwargs={}, response_metadata={}, id='8cd1c36c-96d5-45df-ba96-dfb9c286cfa4'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 93, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a010e5d37f', 'id': 'chatcmpl-EbOq987jIYsiKvGbCGi0wG2HEDLAz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15e3b-a2b1-7124-92e0-bce8ce45a637-0', tool_calls=[{'name': 'send_email_t

In [ ]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused ! Appriving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result : {result['messages'][-1].content}")

paused ! Appriving...
Result : The email has been sent again to john@example.com with the subject "Hello" and the body "This is a test email."


In [ ]:
from langgraph.types import Command
# Rejecting the action instead of approving it

if "__interrupt__" in result:
    print("paused ! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result : {result['messages'][-1].content}")

paused ! Approving...
Result : It seems that the request to send the email was rejected. If you would like to proceed with sending the email to john@example.com with the subject "Hello" and the body "This is a test email." again, please let me know!


In [28]:
result

{'messages': [HumanMessage(content="send email to john@example.com with subject 'Hello' and body 'This is a test email.'", additional_kwargs={}, response_metadata={}, id='8cd1c36c-96d5-45df-ba96-dfb9c286cfa4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 93, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a010e5d37f', 'id': 'chatcmpl-EbOq987jIYsiKvGbCGi0wG2HEDLAz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15e3b-a2b1-7124-92e0-bce8ce45a637-0', tool_calls=[{'name': 'send_email_tool', 'ar

In [37]:
config = {"configurable" : {"thread_id": "test-8"} }
result = agent.invoke(
    {"messages":[HumanMessage(
    content="Edit email to john@example.com with subject 'Hello' and body 'This is a test email.'")]},
    config=config
    )
print(f"Response : {result}")


from langgraph.types import Command
# Rejecting the action instead of approving it

if "__interrupt__" in result:
    print("paused ! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name": "send_email_tool",
                            "args" :{
                                "recipient": "correct@gmail.com",
                                "subject": "Corrected Subject", 
                                "body": "This is the corrected email body."
                            }
                     }
                }
                ]
            }
        ),
        config=config
    )

    print(f"Result : {result['messages'][-1].content}")

Response : {'messages': [HumanMessage(content="Edit email to john@example.com with subject 'Hello' and body 'This is a test email.'", additional_kwargs={}, response_metadata={}, id='a667e53f-0f1c-4a47-9e97-400c4a6fe0d2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 93, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a010e5d37f', 'id': 'chatcmpl-EbP2cH7NjZn0SuRACXNhn6tsc3meR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15e47-707e-740e-b49d-d46cf05e867a-0', tool_calls=[{'name': 'read_email_t

In [ ]:
config = {"configurable" : {"thread_id": "test-5"} }
# Interrupted tool executions solution - pending

from langgraph.types import Command
# Rejecting the action instead of approving it

if "__interrupt__" in result:
    print("paused ! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result : {result['messages'][-1].content}")

paused ! Approving...


OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_4jJzGODYmbuweuNpZXnQSEgO", 'type': 'invalid_request_error', 'param': 'messages.[4].role', 'code': None}}